<a href="https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [13]:

date_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

print(date_check.to_string(index=False))


 row_count first_date  last_date
   9841378 2026-03-01 2026-03-31


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [15]:

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

field_check = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_gsc_clicks,
        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS missing_ga4_sessions
    FROM read_parquet('{march_path}')
""").df()

print(field_check.to_string(index=False))


 row_count  gsc_available_rows  ga4_available_rows  missing_gsc_clicks  missing_ga4_sessions
   9841378             3611061              413966                   0               3018741


- **Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`. These are observed performance signals, provided the relevant data is available.
- **Label:** Future performance outcome; not yet defined or verified.
- **Context:** `report_date`, `month`, `client_hash_id`, `content_hash_id`, `gsc_data_available`, `ga4_data_available`.
- **Excluded:** Future-period performance from features, because it would not be known at the decision moment and could cause data leakage.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

availability_check = con.execute(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_clicks IS NOT NULL
        ) AS usable_gsc_click_rows
    FROM read_parquet('{march_path}')
""").df()

print(availability_check.to_string(index=False))



 gsc_available_rows  ga4_available_rows  usable_gsc_click_rows
            3611061              413966                3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:

limits_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS missing_ga4_sessions
    FROM read_parquet('{march_path}')
""").df()

print(limits_check.to_string(index=False))


 total_rows  gsc_available_rows  ga4_available_rows  missing_ga4_sessions
    9841378             3611061              413966               3018741


- This data shows observed associations, not proof that refreshing content causes more clicks.
- Client histories are unbalanced, so not every client has the same amount of historical data.
- GSC and GA4 availability differs; missing GA4 values must not be treated as zero.
- The March 2026 slice alone cannot establish future performance.
- Overlapping time windows may introduce leakage if future information is included in features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.